# Cloud-Native Anomaly Detection — Transformer Model

Multivariate time-series anomaly detection using a Transformer encoder.  
Trained locally, tracked with MLflow, containerised with Docker, and deployable to GCP Vertex AI.

**Stack:** Python · PyTorch · MLflow · Docker · GCP Vertex AI

---
## Architecture
```
Multivariate Time-Series
        ↓
  Sliding Window → Patch Embedding
        ↓
  Transformer Encoder (multi-head attention)
        ↓
  Reconstruction Head
        ↓
  Reconstruction Error → Anomaly Score → Threshold → Flag
```

## 1. Install Dependencies

In [ ]:
# !pip install torch mlflow scikit-learn pandas numpy matplotlib

## 2. Imports & Config

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
import mlflow
import mlflow.pytorch

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
SEQ_LEN    = 32      # sliding window length
N_FEATURES = 4       # number of input channels
D_MODEL    = 64      # transformer hidden dim
N_HEADS    = 4
N_LAYERS   = 2
BATCH_SIZE = 64
EPOCHS     = 30
LR         = 1e-3
THRESHOLD_PERCENTILE = 95   # anomaly score threshold

print(f"Device: {DEVICE}")

## 3. Synthetic Multivariate Data

In [ ]:
def make_multivariate_data(n=2000, n_features=4, anomaly_frac=0.05, seed=42):
    np.random.seed(seed)
    t = np.linspace(0, 8 * np.pi, n)
    data = np.column_stack([
        np.sin(t) + np.random.normal(0, 0.05, n),
        np.cos(t) + np.random.normal(0, 0.05, n),
        0.5 * np.sin(2 * t) + np.random.normal(0, 0.05, n),
        np.random.normal(0, 0.1, n),
    ])
    labels = np.zeros(n, dtype=int)
    n_anomalies = int(n * anomaly_frac)
    anom_idx = np.random.choice(n, n_anomalies, replace=False)
    data[anom_idx] += np.random.uniform(2.5, 4.0, (n_anomalies, n_features))
    labels[anom_idx] = 1
    return data.astype(np.float32), labels

raw_data, labels = make_multivariate_data()

scaler = StandardScaler()
data_scaled = scaler.fit_transform(raw_data)

# Train on normal data only (unsupervised anomaly detection)
normal_idx = np.where(labels == 0)[0]
train_data  = data_scaled[normal_idx[:1200]]
test_data   = data_scaled
test_labels = labels

print(f"Train: {train_data.shape}  |  Test: {test_data.shape}  |  Anomaly rate: {labels.mean():.2%}")

## 4. Dataset & DataLoader

In [ ]:
class TimeSeriesDataset(Dataset):
    def __init__(self, data, seq_len):
        self.data    = torch.tensor(data, dtype=torch.float32)
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        window = self.data[idx : idx + self.seq_len]   # (seq_len, n_features)
        return window, window   # input == target for reconstruction

train_ds = TimeSeriesDataset(train_data, SEQ_LEN)
test_ds  = TimeSeriesDataset(test_data,  SEQ_LEN)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)
print(f"Train batches: {len(train_dl)}  |  Test batches: {len(test_dl)}")

## 5. Transformer Autoencoder Model

In [ ]:
class TransformerAnomalyDetector(nn.Module):
    """
    Reconstruction-based anomaly detector.
    High reconstruction error → anomaly.
    """
    def __init__(self, n_features, d_model, n_heads, n_layers, seq_len, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.pos_embed  = nn.Embedding(seq_len, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.decoder = nn.Linear(d_model, n_features)

    def forward(self, x):
        # x: (batch, seq_len, n_features)
        positions = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        x_proj    = self.input_proj(x) + self.pos_embed(positions)
        encoded   = self.encoder(x_proj)
        recon     = self.decoder(encoded)
        return recon

model = TransformerAnomalyDetector(
    n_features=N_FEATURES, d_model=D_MODEL,
    n_heads=N_HEADS, n_layers=N_LAYERS, seq_len=SEQ_LEN
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {n_params:,}")
print(model)

## 6. Training with MLflow Tracking

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.MSELoss()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

mlflow.set_experiment("cloud-anomaly-detection")

with mlflow.start_run(run_name="transformer-autoencoder"):
    mlflow.log_params({
        "seq_len": SEQ_LEN, "d_model": D_MODEL, "n_heads": N_HEADS,
        "n_layers": N_LAYERS, "lr": LR, "epochs": EPOCHS, "batch_size": BATCH_SIZE
    })

    train_losses = []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        epoch_loss = 0.0
        for x, y in train_dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            recon = model(x)
            loss  = criterion(recon, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_dl)
        train_losses.append(avg_loss)
        scheduler.step()

        mlflow.log_metric("train_loss", avg_loss, step=epoch)
        if epoch % 5 == 0:
            print(f"Epoch {epoch:3d}/{EPOCHS} | Loss: {avg_loss:.5f}")

    mlflow.pytorch.log_model(model, "model")
    print("Model logged to MLflow.")

## 7. Anomaly Scoring & Evaluation

In [ ]:
model.eval()
recon_errors = []

with torch.no_grad():
    for x, _ in test_dl:
        x     = x.to(DEVICE)
        recon = model(x)
        err   = ((x - recon) ** 2).mean(dim=(1, 2)).cpu().numpy()
        recon_errors.extend(err)

recon_errors = np.array(recon_errors)
threshold    = np.percentile(recon_errors, THRESHOLD_PERCENTILE)

# Align labels (dataset drops first seq_len samples)
aligned_labels = test_labels[SEQ_LEN:]
min_len = min(len(recon_errors), len(aligned_labels))
preds   = (recon_errors[:min_len] > threshold).astype(int)
true    = aligned_labels[:min_len]

print(f"Threshold (p{THRESHOLD_PERCENTILE}): {threshold:.4f}")
print(classification_report(true, preds, target_names=["Normal", "Anomaly"]))
print(f"ROC-AUC: {roc_auc_score(true, recon_errors[:min_len]):.4f}")

## 8. Visualise Results

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

axes[0].plot(recon_errors[:min_len], linewidth=0.7, label="Reconstruction Error")
axes[0].axhline(threshold, color="red", linestyle="--", label=f"Threshold (p{THRESHOLD_PERCENTILE})")
axes[0].set_ylabel("MSE")
axes[0].set_title("Anomaly Score vs Threshold")
axes[0].legend()

axes[1].fill_between(range(min_len), true, alpha=0.4, color="orange", label="True Anomaly")
axes[1].fill_between(range(min_len), preds, alpha=0.4, color="red", label="Predicted Anomaly")
axes[1].set_ylabel("Anomaly Flag")
axes[1].set_title("True vs Predicted Anomalies")
axes[1].legend()

plt.tight_layout()
plt.savefig("anomaly_detection_results.png", dpi=150)
plt.show()

## 9. Docker & GCP Vertex AI Deployment Reference

### Dockerfile
```dockerfile
FROM python:3.10-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 8080
CMD ["uvicorn", "serve:app", "--host", "0.0.0.0", "--port", "8080"]
```

### Build & Push to GCP
```bash
docker build -t gcr.io/YOUR_PROJECT/anomaly-detector:v1 .
docker push gcr.io/YOUR_PROJECT/anomaly-detector:v1
```

### Deploy to Vertex AI
```python
from google.cloud import aiplatform

aiplatform.init(project="YOUR_PROJECT", location="us-central1")

model = aiplatform.Model.upload(
    display_name="anomaly-detector",
    artifact_uri="gs://YOUR_BUCKET/model/",
    serving_container_image_uri="gcr.io/YOUR_PROJECT/anomaly-detector:v1",
)
endpoint = model.deploy(machine_type="n1-standard-4")
```